# 2 · Hasard et statistiques de base — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch02_stats/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « pourquoi ? » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch02_stats/03_notebook.ipynb`, créée par `python tools/start_chapter.py 2`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 2.1 à 2.7 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 2.13 | Tendances centrales : `mean`, `median`, `mode` | 🔨 | ★★ | 30 |
|  | 2.14 | Graine fixée ou graine libre ? | 🔮 | ★ | 10 |
|  | 2.15 | Dispersion : `variance`, `std`, `percentile`, `zscore` | 🔨 | ★★★ | 40 |
|  | 2.16 | Un histogramme fait maison | 🔨 | ★★ | 20 |
|  | 2.17 | Galerie des lois usuelles | 🎨 | ★★ | 20 |
|  | 2.18 | 68-95-99,7 : la théorie face aux tirages et aux manchots | 🔬 | ★★ | 20 |
| B | 2.19 | La roue de la fortune : tirer dans une distribution discrète | 🔨 | ★★ | 25 |
|  | 2.20 | Le pelage des animaux : une variable qui dépend d'une autre | 🔮 | ★★ | 15 |
|  | 2.21 | Tirer avec ou sans remise | 🔨 | ★★ | 20 |
| C | 2.22 | Bootstrap : distribution et intervalle de confiance | 🔨 | ★★ | 30 |
|  | 2.23 | Bootstraps de 20 (livre) ou de n (aujourd'hui) ? | 🔬 | ★★ | 30 |
|  | 2.24 | Comparer avec `scipy.stats.bootstrap` | 📦 | ★★ | 15 |
| D | 2.25 | Distances entre chiffres dans l'espace à 784 dimensions | 📦 | ★★ | 25 |
|  | 2.26 | Covariance et corrélation | 🔨 | ★★ | 25 |
|  | 2.27 | Deviner la corrélation d'un nuage de points | 📈 | ★★ | 20 |
|  | 2.28 | Matrices de covariance et de corrélation des manchots | 🔨 | ★★ | 25 |
|  | 2.29 | Le piège de ddof : NumPy, pandas et toi | 🐛 | ★★ | 20 |
|  | 2.30 | Le quartet d'Anscombe | 🎨 | ★★ | 25 |
|  | 2.31 | Docstring et test pytest pour `zscore` | 🛠️ | ★★ | 30 |
|  | 2.32 | Mêmes statistiques, autre dessin : fabrique ton quartet | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")  # your own library

## Objectifs et rappel express

- Résumer des données par un centre (moyenne, médiane, mode) et une dispersion (variance, écart-type, percentiles), et savoir lequel choisir.
- Tirer au hasard de façon reproductible, dans une loi discrète, avec ou sans remise.
- Mesurer l'incertitude d'une statistique par le bootstrap.
- Calculer et interpréter covariance et corrélation, et toujours regarder les données.

**Rappel express.** $\bar{x} = \frac{1}{n}\sum_i x_i$ ; $\mathrm{Var}(x) = \frac{1}{n - \mathrm{ddof}}\sum_i (x_i - \bar{x})^2$ et $\sigma = \sqrt{\mathrm{Var}(x)}$ ; $z_i = \frac{x_i - \bar{x}}{\sigma}$ ; loi normale : 68 %, 95 %, 99,7 % des tirages à moins de 1, 2, 3 écarts-types de la moyenne ; $\mathbb{E}[X] = \sum_k x_k\,p_k$ ; $\mathrm{Cov}(x, y) = \frac{1}{n - \mathrm{ddof}}\sum_i (x_i - \bar{x})(y_i - \bar{y})$ et $r = \frac{\mathrm{Cov}(x, y)}{\sigma_x\,\sigma_y}$. En NumPy : `rng = np.random.default_rng(seed)`, `x.var(ddof=...)`, `np.percentile`, `np.cov(x, y)`, `np.corrcoef(x, y)`.

## Partie 0 · Vérifier tes exercices papier (✏️ de 2.1 à 2.7)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[7, -2]` ou `"texte"`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande. Les réponses pas encore remplies affichent ⏳. L'exercice ∂ 2.8 se corrige avec `05_solutions.md`.

**Ex 2.1 — Moyenne, médiane et mode d'une liste de salaires**

In [ ]:
# Ex 2.1: the values you found on paper
answer_2_1a = ...  # a) the mean salary, rounded to the euro
answer_2_1b = ...  # b) the median
answer_2_1c = ...  # c) the mode
answer_2_1d = ...  # d) how many employees earn LESS than the mean
answer_2_1e = ...  # e) the new mean with 30 000 instead of 12 000, rounded to the euro
answer_2_1f = ...  # f) the new median

for letter, answer in zip("abcdef", [answer_2_1a, answer_2_1b, answer_2_1c, answer_2_1d, answer_2_1e, answer_2_1f]):
    wb.check(f"2.1{letter}", answer)

**Ex 2.2 — De la casse de voitures à la distribution de probabilité**

In [ ]:
# Ex 2.2: the values you found on paper
answer_2_2a = ...  # a) P(minivan), 2 decimals
answer_2_2b = ...  # b) the 5 probabilities in the order of the statement, a list
answer_2_2c = ...  # c) the angle of the minivan slice, in degrees (1 decimal)
answer_2_2d = ...  # d) P(not an SUV), 2 decimals
answer_2_2e = ...  # e) the mean number of pick-ups in 50 draws
answer_2_2f = ...  # f) the cumulative sums of the probabilities, a list
answer_2_2g = ...  # g) the car type chosen by u = 0.71 (a string)

for letter, answer in zip("abcdefg", [answer_2_2a, answer_2_2b, answer_2_2c, answer_2_2d, answer_2_2e, answer_2_2f, answer_2_2g]):
    wb.check(f"2.2{letter}", answer)

**Ex 2.3 — La règle 68-95-99,7 sur les nageoires des manchots**

In [ ]:
# Ex 2.3: the values you found on paper
answer_2_3a = ...  # a) the interval holding about 68 % of the Adelie, a list [start, end] in mm
answer_2_3b = ...  # b) the interval holding about 95 % of them
answer_2_3c = ...  # c) the proportion above 203 mm (3 decimals)
answer_2_3d = ...  # d) the expected number of Adelie above 203 mm among 151 (integer)
answer_2_3e = ...  # e) the z-score of a 210 mm flipper (2 decimals)

for letter, answer in zip("abcde", [answer_2_3a, answer_2_3b, answer_2_3c, answer_2_3d, answer_2_3e]):
    wb.check(f"2.3{letter}", answer)

**Ex 2.4 — Variance : diviser par N ou par N − 1 ?**

In [ ]:
# Ex 2.4: the values you found on paper
answer_2_4a = ...  # a) the mean
answer_2_4b = ...  # b) the sum of the squared deviations
answer_2_4c = ...  # c) the variance with ddof = 0
answer_2_4d = ...  # d) the variance with ddof = 1
answer_2_4e = ...  # e) the standard deviation with ddof = 0 (3 decimals)
answer_2_4f = ...  # f) the standard deviation with ddof = 1 (3 decimals)
answer_2_4g = ...  # g) the mean of the squares
answer_2_4h = ...  # h) the variance (ddof = 0) after adding 100 to every value
answer_2_4i = ...  # i) the standard deviation (ddof = 0) in milliseconds (1 decimal)

for letter, answer in zip("abcdefghi", [answer_2_4a, answer_2_4b, answer_2_4c, answer_2_4d, answer_2_4e, answer_2_4f, answer_2_4g, answer_2_4h, answer_2_4i]):
    wb.check(f"2.4{letter}", answer)

**Ex 2.5 — Espérances : Bernoulli, multinoulli et jeu de hasard**

In [ ]:
# Ex 2.5: the values you found on paper
answer_2_5a = ...  # a) E[X] for the coin (heads = 1)
answer_2_5b = ...  # b) Var(X), 2 decimals
answer_2_5c = ...  # c) the mean number of heads in 200 throws
answer_2_5d = ...  # d) the expected value of a fair 20-sided die
answer_2_5e = ...  # e) the expected class number (1 decimal)
answer_2_5f = ...  # f) the one-hot vector of class 3, a list
answer_2_5g = ...  # g) the limit of the average one-hot vector, a list
answer_2_5h = ...  # h) the mean gain per game, stake deducted (2 decimals)
answer_2_5i = ...  # i) the organiser's mean gain over 400 games, in euros

for letter, answer in zip("abcdefghi", [answer_2_5a, answer_2_5b, answer_2_5c, answer_2_5d, answer_2_5e, answer_2_5f, answer_2_5g, answer_2_5h, answer_2_5i]):
    wb.check(f"2.5{letter}", answer)

**Ex 2.6 — Compter les tirages avec et sans remise**

In [ ]:
# Ex 2.6: the values you found on paper
answer_2_6a = ...  # a) without replacement, order ignored: number of new datasets
answer_2_6b = ...  # b) with replacement, order ignored
answer_2_6c = ...  # c) with replacement, order counted
answer_2_6d = ...  # d) 7 examples, draw 3 without replacement, order ignored
answer_2_6e = ...  # e) number of possible bootstrap sequences of the 5 examples
answer_2_6f = ...  # f) P(a given example is in none of the 5 draws), 4 decimals
answer_2_6g = ...  # g) the same probability for 1 000 draws among 1 000 (4 decimals)
answer_2_6h = ...  # h) the mean share of examples drawn at least once (4 decimals)

for letter, answer in zip("abcdefgh", [answer_2_6a, answer_2_6b, answer_2_6c, answer_2_6d, answer_2_6e, answer_2_6f, answer_2_6g, answer_2_6h]):
    wb.check(f"2.6{letter}", answer)

**Ex 2.7 — Covariance et corrélation de cinq points à la main**

In [ ]:
# Ex 2.7: the values you found on paper
answer_2_7a = ...  # a) the means of x and y, a list
answer_2_7b = ...  # b) the sum of the products of the deviations
answer_2_7c = ...  # c) the covariance (ddof = 0)
answer_2_7d = ...  # d) the covariance with ddof = 1 (2 decimals)
answer_2_7e = ...  # e) the standard deviations of x and y, a list (3 decimals)
answer_2_7f = ...  # f) the correlation r (3 decimals)
answer_2_7g = ...  # g) the covariance with x in minutes
answer_2_7h = ...  # h) the correlation with x in minutes (3 decimals)

for letter, answer in zip("abcdefgh", [answer_2_7a, answer_2_7b, answer_2_7c, answer_2_7d, answer_2_7e, answer_2_7f, answer_2_7g, answer_2_7h]):
    wb.check(f"2.7{letter}", answer)

## Partie A · Résumer des données : centre, dispersion, histogramme, lois

Fiche §2.2 et §2.3. Tu écris les premières fonctions de `mylearn.stats` (centre, dispersion, histogramme) et tu les appliques aux 342 manchots dont les quatre mesures sont connues ; puis tu compares les lois usuelles et la règle 68-95-99,7 à des tirages et à de vraies mesures.

In [ ]:
# Tools and data for the notebook exercises (parts A to D)
import doctest
import inspect
import math
import re

import pytest

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

penguins = wb.datasets.load_penguins()                    # the 344 penguins, a few values missing
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
measured = penguins.dropna(subset=MEASURES).reset_index(drop=True)   # the 342 penguins with their 4 measures
X_measures = measured[MEASURES].to_numpy(copy=True)       # shape (342, 4); copies: the DataFrame stays intact
mass = measured["body_mass_g"].to_numpy(copy=True)        # in grams
flipper = measured["flipper_length_mm"].to_numpy(copy=True)   # in mm
P_CARS = np.array([176, 144, 224, 96, 160]) / 800         # the scrapyard of 2.2: probability of each type
CAR_TYPES = ["berline", "pick-up", "monospace", "SUV", "break"]


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def fr(value, decimals=2):
    """A number written the French way, for the messages: fr(0.25) -> '0,25'."""
    return f"{value:.{decimals}f}".replace(".", ",")


def run_stats_tests(keyword, impl="learner"):
    """Run the tests of mylearn.stats selected by `keyword` (on YOUR code by default)."""
    command = [sys.executable, "-m", "pytest", "tests/test_ch02_stats.py", "-k", keyword, "-q",
               "-p", "no:cacheprovider", "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix("FAILED tests/test_ch02_stats.py::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


def error_name(func, *args, **kwargs):
    """Name of the exception raised by func(*args, **kwargs), or "no error"."""
    try:
        func(*args, **kwargs)
    except NotImplementedError:
        raise
    except Exception as error:  # noqa: BLE001 - we want the name of whatever is raised
        return type(error).__name__
    return "no error"


print(f"{len(penguins)} penguins, {len(measured)} with their 4 measures")

### Ex 2.13 — Tendances centrales : `mean`, `median`, `mode` 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire la moyenne, la médiane et le mode, et voir laquelle résiste à une valeur aberrante.  
**Prérequis :** Ex 2.1 · 0A (NumPy : `axis`, tri, `np.unique`) · fiche §2.2 · **Fil rouge :** Penguins · **mylearn :** `stats.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme en 0A et en 0B) : ouvre `mon_travail/mylearn/stats.py` (créé par `python tools/start_chapter.py 2`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy est permis (`np.asarray`, `.sum()`, `np.sort`, `np.sqrt`, `np.cumsum`…), mais pas la fonction qui ferait tout le travail à ta place : `np.mean`, `np.median`, `np.var`, `np.std`, `np.percentile`, `np.histogram`, `np.cov`, `np.corrcoef` et `rng.choice` sont les **oracles** des tests. Une fonction peut réutiliser tes autres fonctions (`std` appelle `variance`). Chaque fonction commence par convertir et vérifier son entrée (`np.asarray(x, dtype=float)` ; `ValueError` si c'est vide ou s'il y a des NaN, `np.isnan`) : écris une petite fonction d'aide, par exemple `_as_numbers(x)`, et appelle-la dans les fonctions de calcul (pas dans `mode` ni dans `sample`, qui acceptent aussi des chaînes de caractères). La cellule de vérification recharge ta librairie, vérifie quelques valeurs, puis lance les tests de ces fonctions ; `python -m pytest tests/test_ch02_stats.py -q` les lance tous dans un terminal.

Écris `mean`, `median` et `mode`.
- `mean(x, axis=None)` : la somme divisée par le nombre de valeurs. Avec `axis`, une moyenne par ligne ou par colonne : `x.sum(axis=axis)` divisé par `x.shape[axis]` (0A).
- `median(x, axis=None)` : trie (`np.sort(x, axis=...)`), puis prends la valeur du milieu, ou la moyenne des deux valeurs du milieu si leur nombre est pair. Le long d'un axe, `np.take(x_sorted, k, axis=axis)` prend l'élément n° `k` de chaque ligne ou de chaque colonne.
- `mode(x)` : compte chaque valeur (`np.unique(x, return_counts=True)`), puis garde **toutes** celles qui atteignent le plus grand compte, triées (fiche, ⚠️ « Le mode quand tout est à égalité »). Elle doit marcher aussi sur des chaînes de caractères : ne convertis pas en `float` dans `mode`.

Vérifications, sur les 342 manchots de `measured` (la cellule de vérification les calcule avec tes fonctions : il n'y a rien à recopier) :
a) `mean(mass)`, la masse moyenne (1 décimale) ;
b) `median(mass)` ;
c) `mode(mass)` : combien de manchots ont cette masse (la vérification l'affiche) ? Pourquoi une masse aussi « ronde » ?
d) `mean(X_measures, axis=0)` : la moyenne de chacune des 4 mesures ;
e) le `mode` de la colonne `species` ;
f) une erreur de saisie ajoute un manchot de 57 000 g (au lieu de 5 700 g) : de combien la moyenne et la médiane augmentent-elles ? La vérification calcule `[hausse de la moyenne, hausse de la médiane]` avec tes fonctions (1 décimale).

Puis les tests de ces trois fonctions.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.13"):
    st = mylearn.stats
    wb.check("2.13a", st.mean(mass))
    wb.check("2.13b", st.median(mass))
    mode_13 = st.mode(mass)
    wb.check("2.13c", mode_13)
    if np.ndim(mode_13) == 1 and len(mode_13):            # an array of modes, as the docstring says
        print(f"   the mode {mode_13} is the mass of {int(np.sum(mass == mode_13[0]))} penguins")
    wb.check("2.13d", st.mean(X_measures, axis=0), computed=True)
    wb.check("2.13e", st.mode(measured["species"].to_numpy()))
    with_typo = np.append(mass, 57_000)
    wb.check("2.13f", [st.mean(with_typo) - st.mean(mass), st.median(with_typo) - st.median(mass)])
    run_stats_tests("test_mean_ or test_median_ or test_mode_")

### Ex 2.14 — Graine fixée ou graine libre ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir quand deux tirages pseudo-aléatoires redonnent les mêmes nombres.  
**Prérequis :** 0A (`np.random.default_rng`) · fiche §2.2.1 · livre §2.2.1 · **Fil rouge :** synthétique · **Parcours :** C

Chaque cas ci-dessous affiche deux lignes de tirages. **Sans rien exécuter**, prévois pour chacun si les deux lignes seront identiques : `True` (oui, identiques) ou `False` (non, différentes).

```python
# a)
print(np.random.default_rng(7).integers(1, 7, size=10))
print(np.random.default_rng(7).integers(1, 7, size=10))

# b)
rng = np.random.default_rng(7)
print(rng.integers(1, 7, size=10))
print(rng.integers(1, 7, size=10))

# c)
print(np.random.default_rng().integers(1, 7, size=10))
print(np.random.default_rng().integers(1, 7, size=10))

# d)
np.random.seed(0)
print(np.random.default_rng().integers(1, 7, size=10))
np.random.seed(0)
print(np.random.default_rng().integers(1, 7, size=10))

# e)
rng = np.random.default_rng(7)
first, rest = rng.random(3), rng.random(5)
print(np.concatenate([first, rest]))
print(np.random.default_rng(7).random(8))

# f)
rng = np.random.default_rng(7)
noise = rng.normal(size=3)       # a new line added at the top of an experiment...
print(rng.permutation(6))
rng = np.random.default_rng(7)
print(rng.permutation(6))        # ...compared with the old version, without it
```

Écris ton hypothèse (cellule 📝), puis tes six prédictions ; la vérification ne regarde que tes prédictions. Ensuite seulement, exécute l'**Expérience**.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_2_14a = ...   # True if the two lines will be identical, False otherwise
prediction_2_14b = ...
prediction_2_14c = ...
prediction_2_14d = ...
prediction_2_14e = ...
prediction_2_14f = ...

In [ ]:
for letter, prediction in zip("abcdef", [prediction_2_14a, prediction_2_14b, prediction_2_14c,
                                         prediction_2_14d, prediction_2_14e, prediction_2_14f]):
    wb.check(f"2.14{letter}", prediction)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_2_14a, prediction_2_14b, prediction_2_14c, prediction_2_14d, prediction_2_14e, prediction_2_14f]):
    print("⏳ Ex 2.14 : écris d'abord tes six prédictions, puis relance cette cellule.")
else:
    print("a)", np.random.default_rng(7).integers(1, 7, size=10))
    print("  ", np.random.default_rng(7).integers(1, 7, size=10))

    rng = np.random.default_rng(7)
    print("b)", rng.integers(1, 7, size=10))
    print("  ", rng.integers(1, 7, size=10))

    print("c)", np.random.default_rng().integers(1, 7, size=10))
    print("  ", np.random.default_rng().integers(1, 7, size=10))

    np.random.seed(0)
    print("d)", np.random.default_rng().integers(1, 7, size=10))
    np.random.seed(0)
    print("  ", np.random.default_rng().integers(1, 7, size=10))

    rng = np.random.default_rng(7)
    first, rest = rng.random(3), rng.random(5)
    print("e)", np.concatenate([first, rest]).round(3))
    print("  ", np.random.default_rng(7).random(8).round(3))

    rng = np.random.default_rng(7)
    noise = rng.normal(size=3)       # a new line added at the top of an experiment...
    print("f)", rng.permutation(6))
    rng = np.random.default_rng(7)
    print("  ", rng.permutation(6))  # ...compared with the old version, without it

### Ex 2.15 — Dispersion : `variance`, `std`, `percentile`, `zscore` 🔨 ★★★ ⏱️ 40 min
**Objectif :** écrire variance, écart-type, percentiles et z-scores, et les appliquer aux mesures des manchots.  
**Prérequis :** Ex 2.13, Ex 2.4 · fiche §2.3.2 (🧮 ddof, 🧮 percentiles) · **Fil rouge :** Penguins · **mylearn :** `stats.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/stats.py`, mêmes règles qu'en 2.13 (NumPy permis, sauf les fonctions oracles). Enregistre, puis relance la cellule de vérification.

Écris `variance`, `std`, `percentile` et `zscore`.
- `variance(x, ddof=0, axis=None)` : la somme des carrés des écarts **à la moyenne**, divisée par $n - \mathrm{ddof}$ ; `ValueError` si $n - \mathrm{ddof} \le 0$. Calcule d'abord les écarts, puis élève-les au carré : la formule « moyenne des carrés − carré de la moyenne » de 0B est juste sur le papier, mais elle perd des chiffres sur ordinateur quand les valeurs sont grandes et peu dispersées (un test le vérifie). Avec `axis`, garde la dimension réduite pour soustraire la moyenne (`keepdims=True`, ou `np.expand_dims`).
- `std` : la racine de la variance.
- `percentile(x, q, axis=None)` : la méthode de la fiche (🧮 « percentiles et quantiles ») : trie, place le percentile à la position $\frac{q}{100}(n - 1)$, puis interpole entre les deux valeurs voisines. `q` peut être un nombre ou une liste ; refuse un `q` hors de $[0, 100]$.
- `zscore(x, ddof=0, axis=None)` : $(x - \bar{x}) / \sigma$ ; refuse un écart-type nul (données constantes).

Aucune fonction ne modifie ses arguments : `np.asarray(x, dtype=float)` renvoie **le même** tableau quand `x` en est déjà un ; calcule `x - moyenne` dans un nouveau tableau, jamais avec `-=` (0A.56, « vue modifiée »).

Vérifications (la cellule de vérification les calcule avec tes fonctions) :
a) `variance(flipper)`, la variance des nageoires (1 décimale) ;
b) `std(mass, ddof=1)`, l'écart-type des masses avec ddof = 1 (1 décimale) ;
c) `percentile(mass, [25, 50, 75])`, les trois quartiles des masses ;
d) le z-score de la masse du manchot le plus lourd (2 décimales) ;
e) le nombre de manchots dont la masse est à plus de 2 écarts-types de la moyenne ($|z| > 2$) ;
f) le plus grand $|z|$ du tableau `zscore(X_measures, axis=0)` (2 décimales) : une mesure est-elle aberrante, au sens $|z| > 3$ ?

Puis les tests. Dans tes notes : pourquoi faut-il `axis=0` en f ? Que ferait `axis=None` sur ce tableau ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.15"):
    st = mylearn.stats
    wb.check("2.15a", st.variance(flipper))
    wb.check("2.15b", st.std(mass, ddof=1))
    wb.check("2.15c", st.percentile(mass, [25, 50, 75]))
    z_mass = st.zscore(mass)
    wb.check("2.15d", z_mass[np.argmax(mass)])
    wb.check("2.15e", int(np.sum(np.abs(z_mass) > 2)))
    Z_15 = st.zscore(X_measures, axis=0)
    wb.check("2.15f", np.abs(Z_15).max())
    verdict("2.15", np.allclose(Z_15.mean(axis=0), 0) and np.allclose(Z_15.std(axis=0), 1),
            "chaque colonne standardisée a une moyenne 0 et un écart-type 1.",
            "après zscore(X, axis=0), chaque colonne doit avoir une moyenne 0 et un écart-type 1 (ddof = 0).")
    run_stats_tests("test_variance_ or test_std_ or test_percentile_ or test_zscore_")

### Ex 2.16 — Un histogramme fait maison 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire un histogramme (comptages ou densité) et lire la forme d'une distribution.  
**Prérequis :** Ex 2.13 · fiche §2.2 (🧮 densité) · **Fil rouge :** Penguins · **mylearn :** `stats.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/stats.py`, mêmes règles qu'en 2.13 (NumPy permis, sauf les fonctions oracles). Enregistre, puis relance la cellule de vérification.

Écris `histogram(x, bins=10, bin_range=None, density=False)`. On découpe `[low, high]` (par défaut, du minimum au maximum de `x`) en `bins` intervalles de même largeur : les bords sont `np.linspace(low, high, bins + 1)`. Chaque intervalle contient son bord gauche mais pas son bord droit, **sauf le dernier**, qui garde aussi `high`. Les valeurs hors de `[low, high]` sont ignorées.
- Pour trouver l'intervalle de chaque valeur, `np.searchsorted(edges, x, side="right") - 1` donne le numéro de l'intervalle qui commence au dernier bord inférieur ou égal à la valeur ; ramène les valeurs égales à `high` dans le dernier intervalle ; puis compte avec `np.bincount(..., minlength=bins)`.
- La formule directe $\lfloor (x - \text{low}) / \text{largeur} \rfloor$ marche aussi, mais avec des bords comme 0,1 ou 0,3, qui ne tombent pas juste en binaire, un arrondi peut envoyer une valeur posée pile sur un bord dans l'intervalle voisin. Les tests vérifient les valeurs posées sur un bord.
- Avec `density=True`, divise chaque comptage par (nombre de valeurs comptées × largeur) : l'aire totale des barres vaut alors 1 (fiche, 🧮 densité).

La vérification compare ton histogramme à `np.histogram` sur les nageoires (20 intervalles, puis 7 intervalles de 170 à 240 mm, où beaucoup de nageoires tombent pile sur un bord), vérifie l'aire de la densité, trace l'histogramme des nageoires avec **tes** comptages, puis lance les tests. La cellule d'après compare plusieurs nombres d'intervalles : réponds ensuite dans la cellule 📝.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.16"):
    st = mylearn.stats
    counts_16, edges_16 = st.histogram(flipper, bins=20)
    numpy_counts, numpy_edges = np.histogram(flipper, bins=20)
    verdict("2.16", np.array_equal(counts_16, numpy_counts) and np.allclose(edges_16, numpy_edges),
            "mêmes comptages et mêmes bords que np.histogram (20 intervalles).",
            "tes comptages ou tes bords diffèrent de np.histogram(flipper, bins=20) : compare-les.")
    counts_7, _ = st.histogram(flipper, bins=7, bin_range=(170, 240))
    verdict("2.16", np.array_equal(counts_7, np.histogram(flipper, bins=7, range=(170, 240))[0]),
            "bords imposés (170, 180, …, 240) : mêmes comptages que NumPy, bords compris.",
            "avec bin_range=(170, 240), tes comptages diffèrent de NumPy : attention aux valeurs posées pile sur un bord.")
    density_16, edges_d = st.histogram(flipper, bins=20, density=True)
    area_16 = float(np.sum(density_16 * np.diff(edges_d)))
    verdict("2.16", abs(area_16 - 1) < 1e-9, "l'aire des barres de densité vaut 1.",
            f"l'aire des barres de densité vaut {fr(area_16, 4)} : divise par (nombre de valeurs comptées × largeur).")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.stairs(counts_16, edges_16, fill=True, alpha=0.6)
    ax.set(xlabel="flipper_length_mm", ylabel="number of penguins", title="Your histogram of the 342 flippers (20 bins)")
    plt.show()
    run_stats_tests("test_histogram_")

**Combien d'intervalles ?** La cellule suivante trace l'histogramme des nageoires avec **ton** `histogram`, pour 4, 20 et 100 intervalles.

In [ ]:
with wb.attempt("2.16"):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    for ax, n_bins in zip(axes, [4, 20, 100]):
        counts, edges = mylearn.stats.histogram(flipper, bins=n_bins)
        ax.stairs(counts, edges, fill=True, alpha=0.6)
        ax.set(title=f"{n_bins} bins (width {edges[1] - edges[0]:.2f} mm)", xlabel="flipper_length_mm")
    plt.show()

📝 **Mes réponses** (2.16) : avec 4 intervalles, retrouves-tu les deux bosses de la fiche (§2.2) ? Avec 100, pourquoi l'histogramme ressemble-t-il à un peigne, avec tant d'intervalles vides ? Combien d'intervalles choisirais-tu, et pourquoi ?

…

### Ex 2.17 — Galerie des lois usuelles 🎨 ★★ ⏱️ 20 min
**Objectif :** tracer les densités et des tirages des lois uniforme, normale, de Bernoulli et catégorielle.  
**Prérequis :** Ex 2.16 · fiche §2.3.1 à §2.3.4 · livre §2.3 (figures 2.6 à 2.12) · **Fil rouge :** synthétique · **Parcours :** C

Le livre montre chaque loi par sa courbe, puis par des tirages : sur sa figure 2.8, chaque tirage est un point posé sous la courbe, décalé verticalement au hasard pour qu'on distingue les points. Reproduis cette idée en une figure de quatre panneaux (2 × 2) :
1. **uniforme** : les densités des lois uniformes sur $[0, 1]$ et sur $[-1, 1]$ ;
2. **normale** : les densités de quatre lois inspirées de la figure 2.7 du livre (qui ne chiffre pas leurs écarts-types), données dans `NORMALS_17` sous la forme $(\mu, \sigma)$ : $\mathcal{N}(0, 1)$, $\mathcal{N}(1, 1)$, $\mathcal{N}(-1, 0{,}5^2)$ et $\mathcal{N}(-1, 2^2)$, avec **30 tirages** de chacune, dessinés comme des points sous les courbes (une couleur par loi) ;
3. **Bernoulli** : pour une pièce équilibrée ($p = 0{,}5$) et la pièce truquée du livre ($p = 0{,}3$), des bâtons pour les probabilités de 0 et de 1, et à côté les fréquences observées sur 1 000 lancers (figure 2.12) ;
4. **catégorielle** : les probabilités des cinq types de voitures de 2.2 (`P_CARS`, `CAR_TYPES`) et les fréquences observées sur 1 000 tirages (ici, `rng.choice(5, size=1000, p=P_CARS)` ; tu écriras ton propre tirage en 2.19).

Écris d'abord deux fonctions de densité, qui acceptent un array `x` :
- `uniform_pdf(x, a, b)` : $\frac{1}{b - a}$ sur $[a, b]$, 0 ailleurs (`np.where`) ;
- `normal_pdf(x, mu, sigma)` : la formule de la fiche §2.3.2 (`np.exp`, `np.sqrt`, `np.pi`).

Puis complète `draw_gallery(rng)`, qui trace la figure avec ces deux fonctions et **un seul** générateur `rng` (la vérification lui passe `np.random.default_rng(0)`). La vérification compare tes densités à celles de SciPy (`scipy.stats.uniform`, `scipy.stats.norm`) ; la figure, elle, se juge à l'œil : compare-la avec celle du corrigé. Titres, légendes et axes font partie du travail.

In [ ]:
NORMALS_17 = [(0, 1), (1, 1), (-1, 0.5), (-1, 2)]      # (mu, sigma): four normal laws in the spirit of the book's figure 2.7

In [ ]:
def uniform_pdf(x, a, b):
    """Density of the uniform law on [a, b] at x (x: a number or an array)."""
    raise NotImplementedError("uniform_pdf() is not written yet")


def normal_pdf(x, mu, sigma):
    """Density of the normal law N(mu, sigma**2) at x (x: a number or an array)."""
    raise NotImplementedError("normal_pdf() is not written yet")


def draw_gallery(rng):
    """The 2 x 2 figure: uniform, normal (+ 30 draws of each), Bernoulli, categorical."""
    # fig, axes = plt.subplots(2, 2, figsize=(12, 8))    # axes[0, 1] is the top right panel
    # grid = np.linspace(-5, 5, 1001)
    # TODO 1. axes[0, 0]: uniform_pdf on [0, 1] and on [-1, 1]
    # TODO 2. axes[0, 1]: normal_pdf of the 4 laws of NORMALS_17, and 30 draws of each (rng.normal) as dots
    # TODO 3. axes[1, 0]: Bernoulli p = 0.5 and p = 0.3: bars of the probabilities, bars of the frequencies of 1 000 throws
    # TODO 4. axes[1, 1]: P_CARS: bars of the probabilities, bars of the frequencies of 1 000 draws
    raise NotImplementedError("draw_gallery() is not written yet")

In [ ]:
with wb.attempt("2.17"):
    grid_17 = np.linspace(-2.9, 2.9, 59) + 0.013            # avoids the bounds a and b (a convention, see the book)
    verdict("2.17", np.allclose(uniform_pdf(grid_17, -1, 1), scipy_stats.uniform(loc=-1, scale=2).pdf(grid_17))
            and np.allclose(uniform_pdf(grid_17, 0, 1), scipy_stats.uniform().pdf(grid_17)),
            "uniform_pdf coïncide avec scipy.stats.uniform.",
            "uniform_pdf diffère de scipy.stats.uniform : la hauteur vaut 1 / (b − a) sur [a, b], et 0 ailleurs.")
    verdict("2.17", all(np.allclose(normal_pdf(grid_17, mu, sigma), scipy_stats.norm(mu, sigma).pdf(grid_17))
                        for mu, sigma in NORMALS_17),
            "normal_pdf coïncide avec scipy.stats.norm pour les quatre lois.",
            "normal_pdf diffère de scipy.stats.norm(mu, sigma).pdf : relis la formule (le carré, 2σ², la racine de 2π).")
    draw_gallery(np.random.default_rng(0))
    plt.show()

### Ex 2.18 — 68-95-99,7 : la théorie face aux tirages et aux manchots 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer la part des valeurs à moins de 1, 2 et 3 écarts-types de la moyenne, sur des tirages normaux puis sur de vraies mesures.  
**Prérequis :** Ex 2.15, Ex 2.3 · fiche §2.3.2 · livre §2.3.2 (figures 2.10 et 2.11) · **Fil rouge :** Penguins · **Parcours :** M, C

Écris `share_within(x, k)` : la proportion des valeurs de `x` à **strictement moins** de `k` écarts-types de leur moyenne, $|x_i - \bar{x}| < k\,\sigma$, avec l'écart-type de ddof = 0 (tu peux utiliser NumPy ici). Puis :
a) `shares_normal` : la liste des trois proportions, pour $k = 1, 2, 3$, sur 100 000 tirages `np.random.default_rng(0).standard_normal(100_000)` (3 décimales) ;
b) `shares_adelie` : les mêmes proportions pour les nageoires des 151 Adélie (`adelie_flipper`) : le modèle normal de 2.3 tient-il ?
c) `n_above_203` : combien d'Adélie ont une nageoire de **strictement plus** de 203 mm ? Compare avec ta réponse à 2.3 d ;
d) `shares_all` : les trois proportions pour les nageoires de **tous** les manchots (`flipper`).

La cellule suivante trace les deux histogrammes (en densité) avec la courbe normale de même moyenne et de même écart-type. Réponds ensuite dans la cellule 📝 : pour quelles données la règle marche-t-elle ? Pourquoi échoue-t-elle sur les nageoires de tous les manchots, et dans quel sens ?

In [ ]:
adelie_flipper = measured.loc[measured["species"] == "Adelie", "flipper_length_mm"].to_numpy()   # 151 Adelie


def plot_normal_fit(ax, x, title):
    """Density histogram of x, and the normal density with the same mean and standard deviation (ddof=0)."""
    ax.hist(x, bins=20, density=True, alpha=0.6, label="data")
    grid = np.linspace(x.min() - 10, x.max() + 10, 300)
    ax.plot(grid, scipy_stats.norm(x.mean(), x.std()).pdf(grid), label="normal, same mean and std")
    ax.set(title=title, xlabel="flipper_length_mm", ylabel="density")
    ax.legend()

In [ ]:
def share_within(x, k):
    """Proportion of the values of x strictly less than k standard deviations (ddof=0) away from their mean."""
    raise NotImplementedError("share_within() is not written yet")


shares_normal = ...   # a) [k = 1, k = 2, k = 3] for 100 000 standard normal draws (seed 0)
shares_adelie = ...   # b) the same for adelie_flipper
n_above_203 = ...     # c)
shares_all = ...      # d) the same for flipper (all the penguins)

In [ ]:
wb.check("2.18a", shares_normal)
wb.check("2.18b", shares_adelie)
wb.check("2.18c", n_above_203)
wb.check("2.18d", shares_all)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
plot_normal_fit(axes[0], adelie_flipper, "Adelie flippers (151)")
plot_normal_fit(axes[1], flipper, "All the flippers (342)")
plt.show()

📝 **Mes réponses** (2.18) : pour quelles données la règle marche-t-elle ? Pourquoi échoue-t-elle sur tous les manchots, et dans quel sens ?

…

## Partie B · Tirer au hasard : roue, dépendance, avec ou sans remise

Fiche §2.3.4, §2.4 et §2.5. Tu programmes la roue de loterie de la fiche (un tirage dans une loi catégorielle), tu simules une variable qui dépend d'une autre, puis les tirages avec et sans remise, qui serviront au bootstrap.

### Ex 2.19 — La roue de la fortune : tirer dans une distribution discrète 🔨 ★★ ⏱️ 25 min
**Objectif :** tirer dans une loi catégorielle en inversant les sommes cumulées, et comparer les fréquences obtenues aux probabilités.  
**Prérequis :** Ex 2.2, Ex 2.16 · fiche §2.3.4 · livre §2.2 (figures 2.2 à 2.4) · **Fil rouge :** synthétique · **mylearn :** `stats.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/stats.py`, mêmes règles qu'en 2.13 (NumPy permis, sauf les fonctions oracles). Enregistre, puis relance la cellule de vérification.

Écris `sample_categorical(p, size=None, rng=None)` en suivant **exactement** l'algorithme de sa docstring : c'est la roue de la fiche (§2.3.4) et le calcul de 2.2 f et g.
1. Vérifie `p` : une dimension, aucune valeur négative, une somme égale à 1 à $10^{-8}$ près ; sinon, `ValueError`.
2. Si `rng` vaut `None`, crée `np.random.default_rng()` (sans graine : pas reproductible).
3. Tire `u = rng.random(size)`, puis cherche le segment de chaque `u` : `np.searchsorted(np.cumsum(p), u, side="right")`.
4. Ramène le résultat à `len(p) - 1` au plus (`np.minimum`) : à cause des arrondis, la dernière somme cumulée peut valoir `0.9999999999999999`, et un `u` plus grand donnerait une catégorie qui n'existe pas.
5. Avec `size=None`, renvoie un `int` Python (`int(...)`), sinon un array d'entiers.

La vérification tire 10 000 voitures dans la casse de 2.2 (`P_CARS`) avec la graine 0, compare les fréquences observées aux probabilités, vérifie qu'un tirage unique renvoie un `int`, trace les deux en bâtons, puis lance les tests : ils vérifient aussi qu'à graine égale, tes tirages sont **exactement** ceux de l'algorithme documenté.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.19"):
    st = mylearn.stats
    cars_19 = np.asarray(st.sample_categorical(P_CARS, size=10_000, rng=np.random.default_rng(0)))
    freq_19 = np.bincount(cars_19, minlength=len(P_CARS)) / len(cars_19)
    gap_19 = float(np.abs(freq_19 - P_CARS).max())
    verdict("2.19", cars_19.shape == (10_000,) and gap_19 < 0.02,
            f"10 000 tirages, des fréquences proches des probabilités (écart maximal {fr(gap_19, 3)}).",
            f"écart maximal de {fr(gap_19, 3)} entre fréquences et probabilités (ou pas 10 000 tirages) : relis l'algorithme.")
    one_19 = st.sample_categorical(P_CARS, rng=np.random.default_rng(1))
    verdict("2.19", isinstance(one_19, int), "un tirage unique (size=None) renvoie un int.",
            f"avec size=None, renvoie un int Python (reçu : {type(one_19).__name__}).")
    positions_19 = np.arange(len(P_CARS))
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(positions_19 - 0.2, P_CARS, width=0.4, label="probability (2.2)")
    ax.bar(positions_19 + 0.2, freq_19, width=0.4, label="frequency, 10 000 draws")
    ax.set_xticks(positions_19, CAR_TYPES)
    ax.legend()
    plt.show()
    run_stats_tests("test_sample_categorical_")

### Ex 2.20 — Le pelage des animaux : une variable qui dépend d'une autre 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir puis simuler une variable dont la loi dépend d'une autre, et voir ce que change l'indépendance.  
**Prérequis :** Ex 2.19 · fiche §2.3.5, §2.4, §2.4.1 · livre §2.4 · **Fil rouge :** synthétique · **Parcours :** C

Le livre (§2.4) tire d'abord un animal, puis la longueur de son pelage dans la loi propre à cet animal. Voici le modèle :

| Animal | Probabilité | Pelage (cm) |
|---|---|---|
| chien | 0,5 | loi normale de moyenne 7 et d'écart-type 0,8 |
| chat | 0,3 | loi normale de moyenne 3 et d'écart-type 0,4 |
| hamster | 0,2 | loi normale de moyenne 1 et d'écart-type 0,2 |

**Sans rien exécuter**, prévois :
a) `pred_mean_fur` : la longueur moyenne du pelage sur un très grand nombre d'animaux tirés (1 décimale) ; pense à l'espérance (fiche §2.3.5) ;
b) `pred_bumps` : le nombre de bosses de l'histogramme de tous les pelages (un entier) ;
c) `pred_above_7` : la probabilité qu'un animal tiré au hasard ait un pelage de plus de 7 cm (2 décimales) ;
d) `pred_above_7_dog` : la même probabilité, **sachant** que l'animal tiré est un chien (2 décimales).

Comparer c et d, c'est voir la dépendance : connaître l'animal change la loi du pelage. Écris ton hypothèse, puis tes quatre prédictions ; ensuite seulement, exécute l'**Expérience** : elle tire 20 000 animaux avec ton `sample_categorical` (2.19), puis leurs pelages.

In [ ]:
ANIMALS = ["dog", "cat", "hamster"]
P_ANIMALS = np.array([0.5, 0.3, 0.2])        # step 1: which animal?
FUR_MEAN = np.array([7.0, 3.0, 1.0])         # step 2: its fur length (cm), drawn from the normal law of its animal
FUR_STD = np.array([0.8, 0.4, 0.2])
animal_20 = fur_20 = None                    # filled by the experiment below

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
pred_mean_fur = ...      # a)
pred_bumps = ...         # b) an integer
pred_above_7 = ...       # c)
pred_above_7_dog = ...   # d)

In [ ]:
wb.check("2.20a", pred_mean_fur)
wb.check("2.20b", pred_bumps)
wb.check("2.20c", pred_above_7)
wb.check("2.20d", pred_above_7_dog)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
if any(answer is ... or answer is None for answer in [pred_mean_fur, pred_bumps, pred_above_7, pred_above_7_dog]):
    print("⏳ Ex 2.20 : écris d'abord tes quatre prédictions, puis relance cette cellule.")
else:
    with wb.attempt("2.20"):
        rng_20 = np.random.default_rng(0)
        animal_20 = mylearn.stats.sample_categorical(P_ANIMALS, size=20_000, rng=rng_20)   # step 1: the animals
        fur_20 = rng_20.normal(FUR_MEAN[animal_20], FUR_STD[animal_20])                   # step 2: each fur, given its animal
        print(f"mean fur: {fur_20.mean():.2f} cm · P(fur > 7 cm) ≈ {np.mean(fur_20 > 7):.3f} · "
              f"P(fur > 7 cm | dog) ≈ {np.mean(fur_20[animal_20 == 0] > 7):.3f}")
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        axes[0].hist(fur_20, bins=80, color="0.5")
        axes[0].set_title("All the animals")
        for k, name in enumerate(ANIMALS):
            axes[1].hist(fur_20[animal_20 == k], bins=40, alpha=0.6, label=name)
        axes[1].set_title("By animal")
        axes[1].legend()
        for ax in axes:
            ax.set_xlabel("fur length (cm)")
        plt.show()

e) On mélange maintenant les pelages au hasard, sans toucher aux animaux (`rng.permutation(fur_20)`) : chaque animal reçoit le pelage d'un animal quelconque, et les deux colonnes deviennent **indépendantes**. Prévois, **avant** d'exécuter la cellule d'après, `pred_hamster_shuffled` : la longueur moyenne du pelage des « hamsters » après ce mélange (1 décimale).

In [ ]:
pred_hamster_shuffled = ...   # e)

In [ ]:
wb.check("2.20e", pred_hamster_shuffled)

In [ ]:
if any(answer is ... or answer is None for answer in [pred_hamster_shuffled]):
    print("⏳ Ex 2.20 : écris d'abord ta prédiction e), puis relance cette cellule.")
else:
    if fur_20 is None:
        print("⏳ Ex 2.20 : exécute d'abord l'expérience de a) à d) (elle utilise ton sample_categorical de 2.19).")
    else:
        shuffled_fur_20 = np.random.default_rng(1).permutation(fur_20)     # the furs, in a random order
        print(f"mean fur of the hamsters: {fur_20[animal_20 == 2].mean():.2f} cm before shuffling, "
              f"{shuffled_fur_20[animal_20 == 2].mean():.2f} cm after")
        fig, ax = plt.subplots(figsize=(7, 3.5))
        for k, name in enumerate(ANIMALS):
            ax.hist(shuffled_fur_20[animal_20 == k], bins=40, alpha=0.5, density=True, label=name)
        ax.set(title="After shuffling: the same law for every animal", xlabel="fur length (cm)", ylabel="density")
        ax.legend()
        plt.show()

### Ex 2.21 — Tirer avec ou sans remise 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire les tirages avec et sans remise, et les reconnaître dans les usages du ML (mini-batches, bootstrap).  
**Prérequis :** Quiz 2.Q9 · 0A (mini-batches) · fiche §2.5 · livre §2.5 (figures 2.13 et 2.14) · **Fil rouge :** synthétique · **mylearn :** `stats.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/stats.py`, mêmes règles qu'en 2.13 (NumPy permis, sauf les fonctions oracles). Enregistre, puis relance la cellule de vérification.

Écris `sample(population, size, replace=True, rng=None)` en suivant l'algorithme de sa docstring : convertis la population en array (`np.asarray`), puis choisis des **indices** de lignes :
- **avec remise** : `rng.integers(0, n, size)` (chaque tirage choisit parmi les $n$ éléments) ;
- **sans remise** : les `size` premiers éléments d'une permutation, `rng.permutation(n)[:size]` ;

et renvoie `population[indices]`. Refuse une taille négative, une population vide, ou plus de tirages que d'éléments sans remise (`ValueError`). Si `rng` vaut `None`, crée `np.random.default_rng()`.

Vérifications, avec les huit lettres `LETTERS` (les huit objets des figures 2.13 et 2.14 du livre) ; la cellule de vérification les calcule avec ta fonction :
a) `"".join(sample(LETTERS, 8, rng=np.random.default_rng(1)))` : huit tirages **avec** remise ;
b) le nombre de lettres **distinctes** dans ce tirage ;
c) `"".join(sample(LETTERS, 8, replace=False, rng=np.random.default_rng(1)))` : la même chose **sans** remise ;
d) le nom de l'exception levée par `sample(LETTERS, 10, replace=False)`.

Puis deux usages du ML, à écrire avec **ta** fonction `sample` :
- `epoch_minibatches(n_examples, batch_size, rng)` : les mini-batches d'une epoch (0A), sous forme de **liste** d'arrays : un ordre au hasard des indices `0, …, n_examples − 1`, tiré **sans** remise, puis découpé en morceaux consécutifs de `batch_size` (le dernier peut être plus court). e) La vérification l'appelle pour 10 exemples, des mini-batches de 4 et `rng = np.random.default_rng(0)`, et vérifie le **deuxième** mini-batch.
- `mean_share_distinct(n_resamples, n, rng)` : la part moyenne d'éléments **distincts** dans `n_resamples` rééchantillons de `n` tirages **avec** remise parmi `np.arange(n)`, tous tirés avec le même générateur `rng`. f) La vérification l'appelle pour 200 rééchantillons de 1 000, avec `np.random.default_rng(0)` (3 décimales) : compare avec 2.6 h.

In [ ]:
LETTERS = list("ABCDEFGH")   # the pool of eight objects of the book's figures 2.13 and 2.14

In [ ]:
def epoch_minibatches(n_examples, batch_size, rng):
    """The mini-batches of one epoch: a random order of the indices 0, ..., n_examples - 1, drawn WITHOUT
    replacement with YOUR mylearn.stats.sample, cut into consecutive pieces of batch_size (a list of arrays)."""
    raise NotImplementedError("epoch_minibatches() is not written yet")


def mean_share_distinct(n_resamples, n, rng):
    """Mean share of distinct elements in n_resamples resamples of n draws WITH replacement among np.arange(n)
    (YOUR mylearn.stats.sample, the same generator rng for all the resamples)."""
    raise NotImplementedError("mean_share_distinct() is not written yet")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.21"):
    st = mylearn.stats
    drawn_21 = st.sample(LETTERS, 8, rng=np.random.default_rng(1))
    wb.check("2.21a", "".join(drawn_21))
    wb.check("2.21b", len(set(drawn_21)))
    wb.check("2.21c", "".join(st.sample(LETTERS, 8, replace=False, rng=np.random.default_rng(1))))
    wb.check("2.21d", error_name(st.sample, LETTERS, 10, replace=False))
    run_stats_tests("test_sample_ and not categorical")
with wb.attempt("2.21"):
    batches_21 = [np.asarray(batch) for batch in epoch_minibatches(10, 4, np.random.default_rng(0))]
    verdict("2.21", sorted(np.concatenate(batches_21).tolist()) == list(range(10))
            and [len(batch) for batch in batches_21] == [4, 4, 2],
            "une epoch : chaque exemple une fois, en mini-batches de 4, 4 et 2.",
            "une epoch doit contenir chaque exemple exactement une fois, en mini-batches de 4, 4 et 2.")
    wb.check("2.21e", batches_21[1])
    wb.check("2.21f", mean_share_distinct(200, 1000, np.random.default_rng(0)))

## Partie C · Le bootstrap et son intervalle de confiance

Fiche §2.6. Tu programmes le bootstrap et tu l'appliques à la masse des manchots Chinstrap ; puis tu mets à l'épreuve la recette du livre (des rééchantillons de 20) et tu compares tes intervalles avec l'outil professionnel, `scipy.stats.bootstrap`.

In [ ]:
chinstrap_mass = measured.loc[measured["species"] == "Chinstrap", "body_mass_g"].to_numpy()   # 68 penguins


def plot_bootstrap(boot, estimate, ci, title, xlabel):
    """Histogram of bootstrap values, with the estimate (dashed line) and the interval (shaded)."""
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(boot, bins=30, color="0.6")
    ax.axvspan(ci[0], ci[1], alpha=0.2, label=f"interval [{ci[0]:.1f}, {ci[1]:.1f}]")
    ax.axvline(estimate, ls="--", color="black", label=f"estimate {estimate:.1f}")
    ax.set(title=title, xlabel=xlabel, ylabel="number of resamples")
    ax.legend()
    plt.show()


def check_widths_23(widths):
    """Verdict of 2.23 (widths for SIZES_23)."""
    widths = np.asarray(widths, dtype=float)
    ok = widths.shape == (len(SIZES_23),) and bool(np.all(np.diff(widths) < 0)) and 25 < widths[-1] < 45
    verdict("2.23", ok, "tes largeurs sont mesurées : lis le graphique, puis réponds dans la cellule 📝.",
            "tes largeurs ne sont pas celles attendues : ci_width doit renvoyer la largeur de l'intervalle à 80 %, "
            "avec des rééchantillons de la taille demandée (vérifie confidence et sample_size).")


def check_coverage_23(cover_20, cover_500):
    """Verdict of 2.23 (coverage of the two recipes)."""
    verdict("2.23", cover_20 > 0.95 and 0.65 <= cover_500 <= 0.93,
            "tes deux couvertures sont mesurées : compare-les à la promesse de 80 % (cellule 📝).",
            "couvertures inattendues : relis coverage (à chaque enquête, un NOUVEL échantillon de 500 valeurs tiré "
            "sans remise, puis son intervalle à 80 %).")


def check_n_boot_23(widths):
    """Verdict of 2.23 (widths for 100, 1 000 and 10 000 resamples)."""
    widths = np.asarray(widths, dtype=float)
    ok = len(set(np.round(widths, 9))) == 3 and widths.max() / widths.min() < 1.3
    verdict("2.23", ok, "tu as les trois largeurs : réponds dans la cellule 📝.",
            "ci_width doit utiliser son argument n_boot (et transmettre le même générateur) : relis-la.")

### Ex 2.22 — Bootstrap : distribution et intervalle de confiance 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer le bootstrap et en tirer un intervalle de confiance percentile.  
**Prérequis :** Ex 2.21, Ex 2.15 · 0A (`Callable`, arguments après `*`) · fiche §2.6 (🧮 intervalle de confiance) · livre §2.6 (figures 2.17 et 2.18) · **Fil rouge :** Penguins · **mylearn :** `stats.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/stats.py`, mêmes règles qu'en 2.13 (NumPy permis, sauf les fonctions oracles). Enregistre, puis relance la cellule de vérification.

Écris les deux fonctions du bootstrap.
- `bootstrap_distribution(x, statistic=np.mean, *, n_boot=1000, sample_size=None, rng=None)` : pour chaque rééchantillon, **dans l'ordre**, tire les indices `idx = rng.integers(0, n, size=sample_size)` (avec remise), puis calcule `statistic(x[idx])` ; renvoie les `n_boot` valeurs dans un array. `sample_size=None` veut dire $n$, la taille de `x`. L'étoile `*` de la signature oblige à nommer les arguments qui la suivent (`n_boot=500`, pas `500`) : on ne peut plus les confondre (0A.40). `statistic` est une fonction : `np.mean`, `np.median`, ou la tienne.
- `bootstrap_ci(x, statistic=np.mean, *, confidence=0.95, n_boot=1000, sample_size=None, rng=None)` : réutilise `bootstrap_distribution` avec les mêmes arguments, puis coupe $\frac{1 - c}{2}$ des valeurs de chaque côté, avec $c$ = `confidence` : les bornes sont les percentiles $50\,(1 - c)$ et $50\,(1 + c)$ de la distribution (ta fonction `percentile`). Renvoie un tuple de deux `float` ; refuse `confidence` hors de $]0, 1[$.

Vérifications, sur la masse des 68 manchots Chinstrap (`chinstrap_mass`), chaque appel avec **son** générateur `rng=np.random.default_rng(0)` (la cellule de vérification les calcule avec tes fonctions) :
a) la masse moyenne de l'échantillon (1 décimale) ;
b) l'intervalle de confiance à 95 % de la moyenne, `[bas, haut]` (1 décimale) ;
c) l'intervalle à 80 %, comme dans le livre (figure 2.18) ;
d) l'intervalle à 95 % de la **médiane** (`statistic=np.median`) ;
e) l'écart-type (ddof = 0) de la distribution bootstrap de la moyenne (1 décimale) : c'est l'**erreur type** (*standard error*), la dispersion de la moyenne d'un échantillon à l'autre.

La vérification trace ensuite l'histogramme des 1 000 moyennes, avec la moyenne de l'échantillon et l'intervalle à 95 % (l'équivalent des figures 2.17 et 2.18 du livre), puis lance les tests.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.22"):
    st = mylearn.stats
    wb.check("2.22a", st.mean(chinstrap_mass))
    ci_22 = st.bootstrap_ci(chinstrap_mass, rng=np.random.default_rng(0))
    wb.check("2.22b", ci_22)
    wb.check("2.22c", st.bootstrap_ci(chinstrap_mass, confidence=0.8, rng=np.random.default_rng(0)))
    wb.check("2.22d", st.bootstrap_ci(chinstrap_mass, np.median, rng=np.random.default_rng(0)))
    boot_22 = st.bootstrap_distribution(chinstrap_mass, rng=np.random.default_rng(0))
    wb.check("2.22e", st.std(boot_22))
    plot_bootstrap(boot_22, st.mean(chinstrap_mass), ci_22, "1 000 bootstrap means of the Chinstrap mass", "mean mass (g)")
    run_stats_tests("test_bootstrap_")

### Ex 2.23 — Bootstraps de 20 (livre) ou de n (aujourd'hui) ? 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer l'effet de la taille des rééchantillons sur l'intervalle, et vérifier lequel tient sa promesse.  
**Prérequis :** Ex 2.22 · fiche §2.6 (🕰️ taille des rééchantillons, 🧮 intervalle de confiance) · livre §2.6 · **Fil rouge :** synthétique · **Parcours :** M, C

Le livre (§2.6) part d'une population de 5 000 entiers de 0 à 1 000, en tire un échantillon de 500, puis fait 1 000 rééchantillons de **20** éléments seulement, et lit un intervalle à 80 %. La pratique actuelle prend des rééchantillons de la taille de l'échantillon, ici 500. Refais l'expérience avec la `population` et l'échantillon `sample_23` fournis : on connaît donc la vraie moyenne, `population.mean()`, ce qui n'arrive jamais en vrai.

Écris, avec tes fonctions de `mylearn.stats` :
1. `ci_width(sample_size, rng, n_boot=1000)` : la largeur (haut − bas) de l'intervalle à 80 % de la moyenne de `sample_23`, avec `n_boot` rééchantillons de taille `sample_size` ;
2. `widths_by_size()` : la liste des largeurs pour chaque taille de `SIZES_23` (de 5 à 500), chaque fois avec `np.random.default_rng(1)` ;
3. `coverage(sample_size, n_repeats, rng)` : la proportion d'intervalles à 80 % qui contiennent la vraie moyenne quand on refait **toute** l'enquête `n_repeats` fois. À chaque fois : un nouvel échantillon de 500 valeurs tiré **sans** remise dans `population` (ton `sample`, 2.21), puis son intervalle bootstrap à 80 %, avec 500 rééchantillons de taille `sample_size` ; le même générateur `rng` sert à tout. Un intervalle « à 80 % » honnête doit contenir la vraie valeur environ 80 fois sur 100 (fiche, 🧮) : cette proportion s'appelle la **couverture** (*coverage*) de la méthode.

La vérification affiche les largeurs et le rapport des largeurs (20 contre 500), trace la largeur selon la taille en échelle logarithmique, mesure la couverture des deux recettes, puis compare 100, 1 000 et 10 000 rééchantillons de taille 500. Réponds dans la cellule 📝 :
- De combien l'intervalle du livre est-il trop large ? Comment la largeur varie-t-elle avec la taille des rééchantillons (quand la taille est multipliée par 4) ?
- Laquelle des deux recettes tient sa promesse de 80 % ?
- Augmenter le **nombre** de rééchantillons rétrécit-il l'intervalle ? Que change-t-il ?

In [ ]:
rng_23 = np.random.default_rng(2023)
population = rng_23.integers(0, 1001, size=5000)            # 5 000 integers from 0 to 1 000 (book, §2.6)
sample_23 = population[rng_23.permutation(5000)[:500]]      # a sample of 500 values, drawn without replacement
SIZES_23 = [5, 10, 20, 50, 100, 200, 500]
N_REPEATS_23 = 100 if FAST_MODE else 400                    # number of surveys redone for the coverage
print(f"true mean of the population: {population.mean():.1f} · mean of the sample: {sample_23.mean():.1f}")

In [ ]:
def ci_width(sample_size, rng, n_boot=1000):
    """Width (high - low) of the 80 % bootstrap interval of the mean of sample_23 (n_boot resamples of sample_size)."""
    raise NotImplementedError("ci_width() is not written yet")


def widths_by_size():
    """The list of ci_width(size, np.random.default_rng(1)) for every size of SIZES_23."""
    raise NotImplementedError("widths_by_size() is not written yet")


def coverage(sample_size, n_repeats, rng):
    """Share of n_repeats new surveys (500 values drawn WITHOUT replacement from population) whose 80 % bootstrap
    interval (500 resamples of size sample_size) contains population.mean(). One generator rng for everything."""
    raise NotImplementedError("coverage() is not written yet")

In [ ]:
with wb.attempt("2.23"):
    widths_23 = widths_by_size()
    for size, width in zip(SIZES_23, widths_23):
        print(f"resamples of {size:>3}: width of the 80 % interval = {width:6.1f}")
    ratio_23 = widths_23[SIZES_23.index(20)] / widths_23[SIZES_23.index(500)]
    print(f"ratio of the widths, 20 against 500: {ratio_23:.2f}")
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.loglog(SIZES_23, widths_23, "o-")
    ax.set(xlabel="size of each resample (log scale)", ylabel="width of the 80 % interval (log scale)")
    plt.show()
    check_widths_23(widths_23)
    cover_20 = coverage(20, N_REPEATS_23, np.random.default_rng(7))
    cover_500 = coverage(500, N_REPEATS_23, np.random.default_rng(7))
    print(f"share of the 80 % intervals that contain the true mean: size 20 → {cover_20:.2f} · size 500 → {cover_500:.2f}")
    check_coverage_23(cover_20, cover_500)
    by_n_boot = [ci_width(500, np.random.default_rng(1), n_boot=b) for b in (100, 1000, 10_000)]
    print("width for 100, 1 000 and 10 000 resamples of size 500:", np.round(by_n_boot, 1))
    check_n_boot_23(by_n_boot)

📝 **Mes réponses** (2.23) :

1. …
2. …
3. …

### Ex 2.24 — Comparer avec `scipy.stats.bootstrap` 📦 ★★ ⏱️ 15 min
**Objectif :** utiliser l'outil professionnel du bootstrap en lisant sa documentation, et comparer ses intervalles aux tiens.  
**Prérequis :** Ex 2.22 · fiche §2.6 (🕰️) · documentation de `scipy.stats.bootstrap` · **Fil rouge :** Penguins · **Parcours :** R, C

SciPy fait le bootstrap en une ligne. Lis d'abord sa documentation (`help(scipy_stats.bootstrap)`, ou [en ligne](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html)) : le premier argument, `data`, est une **séquence d'échantillons**, même quand il n'y en a qu'un. On écrit donc `(chinstrap_mass,)`, un tuple d'un seul élément : sans la virgule, SciPy croit recevoir 68 échantillons d'une seule valeur et lève une erreur d'axe (essaie).

Avec `confidence_level=0.95`, `n_resamples=9999` (la valeur par défaut) et, à chaque appel, `rng=np.random.default_rng(0)` :
a) `ci_percentile` : l'intervalle **percentile** (`method="percentile"`) de la moyenne de `chinstrap_mass`, `[bas, haut]`, arrondi au gramme ;
b) `ci_bca` : l'intervalle de la méthode par défaut, `"BCa"` (arrondi au gramme) ;
c) `standard_error` : l'erreur type calculée par SciPy (l'attribut `standard_error` du résultat, 1 décimale) ; compare-la à 2.22 e.

La vérification compare aussi ton intervalle percentile de 2.22 b à celui de SciPy. Dans tes notes : pourquoi ne sont-ils pas identiques, alors que la méthode est la même (compare aussi les 1 000 premières valeurs de `bootstrap_distribution`, l'attribut du résultat de SciPy, avec ta distribution de 2.22) ?

In [ ]:
ci_percentile = ...    # a) [low, high]
ci_bca = ...           # b) [low, high]
standard_error = ...   # c)

In [ ]:
ok_24a = wb.check("2.24a", ci_percentile)
wb.check("2.24b", ci_bca)
wb.check("2.24c", standard_error)
with wb.attempt("2.24"):
    mine_24 = mylearn.stats.bootstrap_ci(chinstrap_mass, rng=np.random.default_rng(0))
    print(f"yours (1 000 resamples): [{mine_24[0]:.1f}, {mine_24[1]:.1f}]")
    if ok_24a:
        gap_24 = max(abs(mine_24[0] - ci_percentile[0]), abs(mine_24[1] - ci_percentile[1]))
        verdict("2.24", gap_24 < 15, f"ton intervalle et celui de SciPy diffèrent d'au plus {fr(gap_24, 1)} g : même méthode, "
                "mais pas le même nombre de rééchantillons.",
                f"ton intervalle et celui de SciPy diffèrent de {fr(gap_24, 1)} g : c'est beaucoup pour la même méthode.")

## Partie D · Grande dimension, covariance et corrélation, ddof, Anscombe

Fiche §2.7 à §2.9. Tu mesures des distances entre images de MNIST, tu écris covariance et corrélation et tu les appliques aux manchots, d'abord tous ensemble puis espèce par espèce, tu traques les pièges de ddof, puis tu redessines le quartet d'Anscombe et tu fabriques le tien.

In [ ]:
bill_length = measured["bill_length_mm"].to_numpy(copy=True)     # in mm (a copy: the DataFrame stays intact)
bill_depth = measured["bill_depth_mm"].to_numpy(copy=True)       # in mm

### Ex 2.25 — Distances entre chiffres dans l'espace à 784 dimensions 📦 ★★ ⏱️ 25 min
**Objectif :** calculer des distances entre images vues comme des points à 784 dimensions, et constater qu'en grande dimension, des points au hasard sont tous à peu près à la même distance.  
**Prérequis :** 0B (distance, 101.3.2) · Ex 2.11 · fiche §2.7 · livre §2.7 · **Fil rouge :** MNIST · **Parcours :** M, C

`X_digits` contient 2 000 images de MNIST aplaties en vecteurs de 784 pixels, entre 0 et 1, et `y_digits` leurs labels. La distance entre deux images est la distance euclidienne de 0B, avec 784 termes : `np.linalg.norm(a - b)`.

a) `dist_01` : la distance entre les images 0 et 1 (2 décimales).
b) `nearest_index` : l'indice (le numéro de ligne dans `X_digits`) de l'image la plus proche de l'image 0, parmi les 1 999 autres. Calcule toutes les distances d'un coup, `np.linalg.norm(X_digits - X_digits[0], axis=1)` (broadcasting, 0A), puis écarte l'image 0 elle-même (sa distance vaut 0 : remplace-la par `np.inf`). Est-ce le même chiffre ? (la vérification affiche les deux labels)
c) `nn_accuracy` : pour chacune des **500 premières** images, cherche la plus proche parmi les 1 999 autres (une boucle sur les 500 images) ; quelle proportion a le même label que sa plus proche voisine (3 décimales) ? C'est déjà un classifieur, celui des plus proches voisins (ch. 13).
d) `same_vs_other` : parmi les 500 premières images, le rapport (distance moyenne entre deux images de chiffres **différents**) / (distance moyenne entre deux images du **même** chiffre), sur toutes les paires d'images distinctes (2 décimales).
e) `contrast_random` : le **contraste** d'un nuage de points est $\frac{d_{\max} - d_{\min}}{d_{\min}}$, où $d_{\min}$ et $d_{\max}$ sont la plus petite et la plus grande distance entre le point 0 et les autres points. Calcule-le pour 1 000 points tirés uniformément dans $[0, 1]^{784}$, `np.random.default_rng(0).random((1000, 784))` (4 décimales).
f) `contrast_mnist` : le contraste des 1 000 premières images de `X_digits` (2 décimales).

La cellule suivante trace le contraste de points tirés au hasard selon la dimension. Dans tes notes : que devient le contraste quand la dimension grandit ? Pourquoi les images de MNIST se comportent-elles autrement que des points tirés au hasard (pense à 2.11, question 6) ?

In [ ]:
X_digits, y_digits = wb.datasets.load_mnist("train", n=2000, flatten=True, normalize=True, seed=0)   # 2 000 images
print(X_digits.shape, X_digits.dtype)

In [ ]:
dist_01 = ...           # a)
nearest_index = ...     # b)
nn_accuracy = ...       # c)
same_vs_other = ...     # d)
contrast_random = ...   # e)
contrast_mnist = ...    # f)

In [ ]:
wb.check("2.25a", dist_01)
if wb.check("2.25b", nearest_index):
    print(f"   label of image 0: {y_digits[0]} · label of its nearest image: {y_digits[nearest_index]}")
wb.check("2.25c", nn_accuracy)
wb.check("2.25d", same_vs_other)
wb.check("2.25e", contrast_random)
wb.check("2.25f", contrast_mnist)

In [ ]:
dims_25 = [1, 2, 3, 10, 30, 100, 300, 784, 3072]
contrasts_25 = []
for dim in dims_25:
    points = np.random.default_rng(1).random((1000, dim))          # 1 000 uniform points in [0, 1]^dim
    d = np.linalg.norm(points[1:] - points[0], axis=1)
    contrasts_25.append((d.max() - d.min()) / d.min())
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.loglog(dims_25, contrasts_25, "o-")
ax.set(xlabel="dimension (log scale)", ylabel="contrast (d_max - d_min) / d_min (log scale)",
       title="Random points: the nearest and the farthest become alike")
plt.show()

### Ex 2.26 — Covariance et corrélation 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire la covariance et la corrélation, et les interpréter sur les mesures des manchots.  
**Prérequis :** Ex 2.7, Ex 2.15 · fiche §2.8.1, §2.8.2 · **Fil rouge :** Penguins · **mylearn :** `stats.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/stats.py`, mêmes règles qu'en 2.13 (NumPy permis, sauf les fonctions oracles). Enregistre, puis relance la cellule de vérification.

Écris `covariance(x, y, ddof=0)` et `correlation(x, y)`.
- `covariance` : la somme des produits des écarts à la moyenne, $(x_i - \bar{x})(y_i - \bar{y})$, divisée par $n - \mathrm{ddof}$ ; `x` et `y` doivent être deux tableaux à une dimension de même longueur (sinon, `ValueError`).
- `correlation` : la covariance divisée par les deux écarts-types, **avec le même ddof** (fiche §2.8.2) ; refuse une variable constante, dont l'écart-type est nul. Les arrondis peuvent donner `1.0000000000000002` : ramène le résultat dans $[-1, 1]$ (`np.clip`).

Aucune fonction ne modifie ses arguments : `np.asarray(x, dtype=float)` renvoie **le même** tableau quand `x` en est déjà un ; calcule `x - moyenne` dans un nouveau tableau, jamais avec `-=` (0A.56, « vue modifiée »).

Vérifications (la cellule de vérification les calcule avec tes fonctions) :
a) la covariance (ddof = 0) de la nageoire (mm) et de la masse (g) (1 décimale) ;
b) leur corrélation (3 décimales) ;
c) la covariance de la nageoire en **cm** et de la masse en **kg** (3 décimales) ; la vérification calcule aussi la corrélation en cm et en kg (2.8, question 6) ;
d) la corrélation de la **longueur** et de l'**épaisseur** du bec, sur tous les manchots (3 décimales). Un bec plus long serait-il plus fin ? Garde ta réponse : tu y reviens en 2.28.

La vérification trace les deux nuages de points de b et de d, puis lance les tests.

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.26"):
    st = mylearn.stats
    wb.check("2.26a", st.covariance(flipper, mass))
    wb.check("2.26b", st.correlation(flipper, mass))
    wb.check("2.26c", st.covariance(flipper / 10, mass / 1000))
    verdict("2.26", abs(st.correlation(flipper / 10, mass / 1000) - st.correlation(flipper, mass)) < 1e-12,
            "en cm et en kg, la corrélation ne change pas.",
            "la corrélation doit rester la même en cm et en kg : utilise le même ddof pour la covariance et les écarts-types.")
    wb.check("2.26d", st.correlation(bill_length, bill_depth))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(flipper, mass, s=10)
    axes[0].set(xlabel="flipper_length_mm", ylabel="body_mass_g", title=f"r = {st.correlation(flipper, mass):.3f}")
    axes[1].scatter(bill_length, bill_depth, s=10)
    axes[1].set(xlabel="bill_length_mm", ylabel="bill_depth_mm", title=f"r = {st.correlation(bill_length, bill_depth):.3f}")
    plt.show()
    run_stats_tests("(test_covariance_ or test_correlation_) and not matrix")

### Ex 2.27 — Deviner la corrélation d'un nuage de points 📈 ★★ ⏱️ 20 min
**Objectif :** estimer une corrélation à l'œil, et repérer les nuages où elle trompe.  
**Prérequis :** Ex 2.26 · fiche §2.8.2 (⚠️ deux contresens) · livre §2.8.2 (figures 2.25 à 2.29) · **Fil rouge :** synthétique · **Parcours :** R, M

Les six nuages A à F ci-dessous ont chacun 60 points, et leurs corrélations sont affichées sous la figure, **dans l'ordre croissant**, pas dans l'ordre des nuages. **Sans rien calculer**, attribue à chaque nuage sa corrélation.

a) `r_guess` : la liste des six corrélations, dans l'ordre des nuages A, B, C, D, E, F (recopie les valeurs affichées) ;
b) `r_without_outlier` : un des nuages doit presque toute sa corrélation à un seul point. Retire ce point (le plus éloigné du centre du nuage) et calcule la corrélation des 59 autres (2 décimales), avec `np.corrcoef` ou ta fonction `correlation`. Les nuages sont dans `clouds_27`, un dictionnaire `{"A": (x, y), …}`.

La cellule **Réponse** affiche ensuite la vraie corrélation de chaque nuage. Puis, dans la cellule 📝 : quel nuage a une corrélation proche de 0 alors que $y$ dépend fortement de $x$ ? Pourquoi la corrélation ne le voit-elle pas ?

In [ ]:
def make_clouds_27():
    """Six clouds of 60 points (x, y), named A to F in a shuffled order: guess by eye, do not read the recipes."""
    rng = np.random.default_rng(97)
    x = rng.normal(0, 1, (6, 60))
    e = rng.normal(0, 1, (6, 60))
    y = np.array([x[0] ** 2 + 0.3 * e[0], -x[1] + 0.45 * e[1], e[2], x[3] + 0.15 * e[3],
                  -0.6 * x[4] + e[4], 0.35 * x[5] + e[5]])
    x[2, 0], y[2, 0] = 9.0, 9.0
    order = rng.permutation(6)
    return {letter: (x[k], y[k]) for letter, k in zip("ABCDEF", order)}


clouds_27 = make_clouds_27()
R_VALUES_27 = sorted(round(float(np.corrcoef(x, y)[0, 1]), 2) for x, y in clouds_27.values())
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (letter, (x, y)) in zip(axes.ravel(), clouds_27.items()):
    ax.scatter(x, y, s=12)
    ax.set_title(letter, fontsize=14)
plt.show()
print("the six correlations, in increasing order (not in the order of the clouds):", R_VALUES_27)

In [ ]:
r_guess = ...             # a) six values, in the order of the clouds A to F
r_without_outlier = ...   # b)

In [ ]:
wb.check("2.27a", r_guess)
wb.check("2.27b", r_without_outlier)

**Réponse** : exécute la cellule pour voir la vraie corrélation de chaque nuage.

In [ ]:
if any(answer is ... or answer is None for answer in [r_guess]):
    print("⏳ Ex 2.27 : donne d'abord ta réponse a), puis relance cette cellule.")
else:
    for letter, (x, y) in clouds_27.items():
        print(f"{letter}: r = {np.corrcoef(x, y)[0, 1]:+.2f}")

📝 **Mes réponses** (2.27) : quel nuage a une corrélation proche de 0 alors que $y$ dépend fortement de $x$ ? Pourquoi la corrélation ne le voit-elle pas ?

…

### Ex 2.28 — Matrices de covariance et de corrélation des manchots 🔨 ★★ ⏱️ 25 min
**Objectif :** calculer les matrices de covariance et de corrélation d'un tableau, et comparer la corrélation de l'ensemble à celle de chaque groupe.  
**Prérequis :** Ex 2.26 · 0B (produit matriciel) · fiche §2.8 (🧮 matrice de covariance) · **Fil rouge :** Penguins · **mylearn :** `stats.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/stats.py`, mêmes règles qu'en 2.13 (NumPy permis, sauf les fonctions oracles). Enregistre, puis relance la cellule de vérification.

Écris `covariance_matrix(X, ddof=0)` et `correlation_matrix(X)`.
- `covariance_matrix` : `X` a une ligne par manchot et une colonne par mesure. Centre chaque colonne (soustrais la moyenne de la colonne), puis la case $(j, k)$ est la somme des produits des colonnes centrées $j$ et $k$, divisée par $n - \mathrm{ddof}$. Deux boucles sur $j$ et $k$ suffisent ; en une ligne, c'est `Xc.T @ Xc / (n - ddof)` (0B : vérifie les formes, $(p, n) \times (n, p)$ donne $(p, p)$).
- `correlation_matrix` : divise chaque case $(j, k)$ de la matrice de covariance par $\sigma_j\,\sigma_k$ ; la diagonale vaut 1.

Vérifications, sur `X_measures` (342 manchots × 4 mesures) :
a) la matrice de corrélation (4 × 4), que la vérification calcule avec ta fonction et affiche avec le nom des mesures ;
b) `most_negative` : d'après cette matrice, la paire de mesures la plus **négativement** corrélée (une liste de deux noms de `MEASURES`).

Ensuite, une question sur les espèces.

In [ ]:
most_negative = ...   # b) a list of two names of MEASURES

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")   # reload your saved file
with wb.attempt("2.28"):
    st = mylearn.stats
    C_28 = st.covariance_matrix(X_measures)
    verdict("2.28", np.shape(C_28) == (4, 4) and np.allclose(np.diag(C_28), X_measures.var(axis=0)),
            "ta matrice de covariance est 4 × 4, avec les variances des colonnes sur la diagonale.",
            "la matrice de covariance doit être 4 × 4, avec les variances des colonnes (ddof = 0) sur la diagonale.")
    R_28 = st.correlation_matrix(X_measures)
    print(pd.DataFrame(R_28, index=MEASURES, columns=MEASURES).round(2))
    wb.check("2.28a", R_28, computed=True)
    run_stats_tests("test_covariance_matrix_ or test_correlation_matrix_")
wb.check("2.28b", most_negative.replace(" ", "").split(",") if isinstance(most_negative, str) else most_negative)

**Et dans chaque espèce ?** En 2.26 d, la corrélation entre la longueur et l'épaisseur du bec est **négative** sur l'ensemble des manchots : un bec plus long serait plus fin.

c) `prediction_gentoo` : **avant de calculer**, prédis le signe de cette corrélation chez les seuls Gentoo : `+1` si elle est positive, `-1` si elle est négative.

In [ ]:
prediction_gentoo = ...   # c) +1 (positive) or -1 (negative), BEFORE computing d

In [ ]:
wb.check("2.28c", prediction_gentoo)

d) Écris `bill_r_by_species()` : la corrélation longueur–épaisseur du bec pour chaque espèce, dans l'ordre Adélie, Chinstrap, Gentoo (une liste, 2 décimales), avec ta `correlation` ou ta `correlation_matrix`. La vérification trace ensuite le nuage coloré par espèce, avec une droite par espèce et une pour l'ensemble.

In [ ]:
def bill_r_by_species():
    """[r Adelie, r Chinstrap, r Gentoo]: correlation of bill length and bill depth within each species."""
    raise NotImplementedError("bill_r_by_species() is not written yet")

In [ ]:
with wb.attempt("2.28"):
    wb.check("2.28d", bill_r_by_species())

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_gentoo]):
    print("⏳ Ex 2.28 : écris d'abord ta prédiction c), puis relance cette cellule.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for species, group in measured.groupby("species"):
        ax.scatter(group["bill_length_mm"], group["bill_depth_mm"], s=12, label=species)
        slope, intercept = np.polyfit(group["bill_length_mm"], group["bill_depth_mm"], 1)
        ends = np.array([group["bill_length_mm"].min(), group["bill_length_mm"].max()])
        ax.plot(ends, intercept + slope * ends)
    slope, intercept = np.polyfit(bill_length, bill_depth, 1)
    ends = np.array([bill_length.min(), bill_length.max()])
    ax.plot(ends, intercept + slope * ends, "k--", label="all the penguins")
    ax.set(xlabel="bill_length_mm", ylabel="bill_depth_mm", title="All the penguins, and each species")
    ax.legend()
    plt.show()

📝 **Mes réponses** (2.28) : compare le signe de la corrélation sur l'ensemble des manchots et dans chaque espèce. Comment l'expliques-tu ?

…

### Ex 2.29 — Le piège de ddof : NumPy, pandas et toi 🐛 ★★ ⏱️ 20 min
**Objectif :** diagnostiquer trois bugs silencieux dus aux conventions de ddof et de `np.cov`, puis les corriger.  
**Prérequis :** Ex 2.28, Ex 2.7 · fiche §2.8 (🕰️ conventions ddof) · pandas (0A) · **Fil rouge :** Penguins · **Parcours :** C

Un collègue a écrit les trois fonctions ci-dessous. Elles tournent sans erreur… mais elles sont fausses. Mesure d'abord les dégâts :
a) `ratio_29` : `colleague_correlation(flipper, mass)` divisée par la vraie corrélation, `np.corrcoef(flipper, mass)[0, 1]` (4 décimales). Reconnais-tu ce nombre (indice : $n = 342$) ? Essaie aussi la fonction sur les cinq étudiants de 2.7 (`hours_2_7`, `grades_2_7`) : que vaut le résultat ?
b) `shape_29` : la forme (`.shape`) du résultat de `colleague_cov_matrix(measured[MEASURES])` ;
c) `std_after_29` : l'écart-type avec ddof = 0 (`np.std`, la convention du `StandardScaler` de scikit-learn) de la colonne `body_mass_g` du tableau standardisé par `colleague_standardize(measured[MEASURES])` (4 décimales).

Puis corrige : écris `correlation_fixed(x, y)`, `cov_matrix_fixed(df)` (la matrice 4 × 4 des covariances des colonnes, ddof = 0) et `standardize_fixed(df)` (chaque colonne de moyenne 0 et d'écart-type 1 avec ddof = 0), avec NumPy ou pandas, sans `mylearn`. Dans tes notes, pour chaque bug : quelle convention a été mélangée, et comment l'aurais-tu repéré ?

In [ ]:
def colleague_correlation(x, y):
    return np.cov(x, y)[0, 1] / (np.std(x) * np.std(y))


def colleague_cov_matrix(df):
    return np.cov(df.to_numpy())


def colleague_standardize(df):
    return (df - df.mean()) / df.std()


hours_2_7 = np.array([1, 2, 3, 4, 5])     # the five students of 2.7: hours of revision
grades_2_7 = np.array([2, 3, 5, 4, 6])    # and grades

In [ ]:
ratio_29 = ...       # a)
shape_29 = ...       # b)
std_after_29 = ...   # c)


def correlation_fixed(x, y):
    """Pearson correlation of x and y."""
    raise NotImplementedError("correlation_fixed() is not written yet")


def cov_matrix_fixed(df):
    """Covariance matrix (ddof=0) of the COLUMNS of df: shape (number of columns, number of columns)."""
    raise NotImplementedError("cov_matrix_fixed() is not written yet")


def standardize_fixed(df):
    """Every column minus its mean, divided by its standard deviation with ddof=0."""
    raise NotImplementedError("standardize_fixed() is not written yet")

In [ ]:
wb.check("2.29a", ratio_29)
wb.check("2.29b", tuple(int(v) for v in re.findall(r"\d+", shape_29)) if isinstance(shape_29, str) else shape_29)
wb.check("2.29c", std_after_29)
with wb.attempt("2.29"):
    verdict("2.29", abs(correlation_fixed(flipper, mass) - np.corrcoef(flipper, mass)[0, 1]) < 1e-12
            and abs(correlation_fixed(hours_2_7, grades_2_7) - np.corrcoef(hours_2_7, grades_2_7)[0, 1]) < 1e-12,
            "correlation_fixed donne la vraie corrélation, aussi pour les cinq étudiants de 2.7.",
            "correlation_fixed doit redonner np.corrcoef, aussi sur les cinq points de 2.7 (une corrélation ne dépasse "
            "jamais 1) : le même ddof en haut et en bas.")
    C_29 = np.asarray(cov_matrix_fixed(measured[MEASURES]), dtype=float)
    verdict("2.29", C_29.shape == (4, 4) and np.allclose(C_29, np.cov(X_measures, rowvar=False, ddof=0)),
            "cov_matrix_fixed : une matrice 4 × 4, celle des colonnes, avec ddof = 0.",
            "cov_matrix_fixed doit renvoyer la matrice 4 × 4 des covariances des COLONNES, avec ddof = 0.")
    Z_29 = np.asarray(standardize_fixed(measured[MEASURES]), dtype=float)
    verdict("2.29", Z_29.shape == (342, 4) and np.allclose(Z_29.mean(axis=0), 0) and np.allclose(Z_29.std(axis=0), 1),
            "standardize_fixed : chaque colonne a une moyenne 0 et un écart-type 1 (ddof = 0).",
            "après standardize_fixed, chaque colonne doit avoir une moyenne 0 et un écart-type 1 avec ddof = 0.")

### Ex 2.30 — Le quartet d'Anscombe 🎨 ★★ ⏱️ 25 min
**Objectif :** redessiner le quartet d'Anscombe avec ses droites de régression, et voir que ses statistiques coïncident… pas toutes.  
**Prérequis :** Ex 2.26, Ex 2.12 · fiche §2.9 · livre §2.9 (figures 2.31 et 2.32) · **Parcours :** M, C

Les quatre jeux d'Anscombe sont dans `ANSCOMBE`, un dictionnaire `{"I": (x, y), …}`. Écris :
- `fit_line(x, y)` : la **droite de régression** de $y$ en $x$, sous la forme `(intercept, slope)`. Sa pente est $a = \frac{\mathrm{Cov}(x, y)}{\mathrm{Var}(x)}$ et son ordonnée à l'origine $b = \bar{y} - a\,\bar{x}$ (formules admises ici, démontrées au ch. 9), avec tes fonctions de `mylearn.stats` ou avec NumPy ;
- `draw_quartet()` : la figure du livre (figure 2.31) : quatre panneaux (2 × 2), chaque nuage avec sa droite, les **mêmes axes** pour les quatre (`sharex=True, sharey=True`), et un titre par panneau.

La vérification contrôle que les quatre jeux ont bien, à 0,01 près, la même moyenne et la même variance de $x$ et de $y$, la même corrélation et la même droite ; puis :
a) `medians_y` : la médiane de $y$ dans chacun des quatre jeux (liste, 2 décimales). Toutes les statistiques sont-elles égales ?
b) `max_residuals` : pour chaque jeu, le plus grand écart $|y_i - (b + a\,x_i)|$ entre un point et **sa** droite (liste, 2 décimales) ;
c) `slope_iii_clean` : la pente de la droite de régression du jeu III privé de son point isolé, celui du plus grand écart (3 décimales). Que vaut alors sa corrélation ?

In [ ]:
x_123 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)
ANSCOMBE = {   # F. J. Anscombe, "Graphs in Statistical Analysis", 1973 (public domain)
    "I": (x_123, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    "II": (x_123, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    "III": (x_123, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    "IV": (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], dtype=float),
           np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}

In [ ]:
def fit_line(x, y):
    """(intercept, slope) of the least-squares line of y on x."""
    raise NotImplementedError("fit_line() is not written yet")


def draw_quartet():
    """Figure 2 x 2: the four sets of ANSCOMBE, each with its line, the same axes everywhere."""
    raise NotImplementedError("draw_quartet() is not written yet")


medians_y = ...         # a)
max_residuals = ...     # b)
slope_iii_clean = ...   # c)

In [ ]:
with wb.attempt("2.30"):
    table_30 = np.array([[np.mean(x), np.mean(y), np.var(x, ddof=1), np.var(y, ddof=1), np.corrcoef(x, y)[0, 1],
                          *fit_line(x, y)] for x, y in ANSCOMBE.values()])
    x_i, y_i = ANSCOMBE["I"]
    verdict("2.30", np.allclose(fit_line(x_i, y_i), np.polyfit(x_i, y_i, 1)[::-1]),
            "fit_line donne la droite des moindres carrés (comme np.polyfit), dans l'ordre (intercept, slope).",
            "fit_line doit renvoyer (intercept, slope), la droite des moindres carrés : vérifie l'ordre et le ddof "
            "(le même pour la covariance et la variance).")
    verdict("2.30", bool(np.all(np.ptp(table_30, axis=0) < 0.01)),
            "les quatre jeux ont les mêmes moyennes, variances, corrélation et droite, à 0,01 près.",
            "les droites (intercept, slope) des quatre jeux devraient coïncider à 0,01 près : vérifie fit_line.")
    draw_quartet()
    plt.show()
wb.check("2.30a", medians_y)
wb.check("2.30b", max_residuals)
wb.check("2.30c", slope_iii_clean)

### Ex 2.31 — Docstring et test pytest pour `zscore` 🛠️ ★★ ⏱️ 30 min
**Objectif :** documenter une fonction au format NumPy et écrire des tests pytest qui attrapent les cas limites d'une standardisation.  
**Prérequis :** Ex 2.15 · 0A.61, 0A.62 · fiche §2.3.2 · **Parcours :** C

Deux gestes de développeur, sur la standardisation de 2.15.

**1. Une docstring.** Écris `flag_outliers(x, threshold=3.0)` : elle renvoie les **indices** (un array d'entiers, dans l'ordre croissant) des valeurs dont le z-score (ddof = 0) dépasse `threshold` en valeur absolue ; calcule les z-scores avec `mylearn.stats.zscore`, et utilise `np.flatnonzero`. Donne-lui une docstring complète au format NumPy (comme en 0A.61) : un résumé d'une ligne, puis les sections `Parameters`, `Returns`, `Raises` (que se passe-t-il si les données sont constantes ?) et `Examples`, avec au moins **deux** exemples `>>>` que doctest vérifie. Pour un exemple, prends des valeurs dont tu peux calculer le résultat à la main : pour `[1, 1, 1, 1, 1, 1, 1, 1, 1, 10]`, calcule d'abord le z-score de 10 (moyenne, écart-type), puis choisis `threshold=2.5`. Pourquoi `threshold=3.0` serait-il un mauvais exemple ?

**2. Des tests.** Écris au moins **trois** tests pytest pour une fonction `zscore(x, ddof=0, axis=None)` qui a le contrat de `mylearn.stats.zscore` :
- une **propriété** : sur des données quelconques, le résultat a une moyenne 0 et un écart-type 1 (`pytest.approx`) ;
- un **cas limite** : des données constantes lèvent une `ValueError` (`pytest.raises`) ;
- l'**axe** : avec `axis=0`, **chaque colonne** d'un tableau 2-D a une moyenne 0 et un écart-type 1.

Range-les dans la liste `my_zscore_tests`. Comme en 0A.62, la vérification lance vraiment pytest : tes tests doivent **passer** sur une version juste (`zscore_ok`) et **échouer** sur chacune des trois versions buggées fournies (lis-les : où est l'erreur ?). Tes tests ne peuvent utiliser que `zscore`, `np`, `math` et `pytest`.

In [ ]:
def zscore_ok(x, ddof=0, axis=None):
    """A correct version, with the contract of mylearn.stats.zscore."""
    x = np.asarray(x, dtype=float)
    std = x.std(axis=axis, ddof=ddof, keepdims=True)
    if x.size == 0 or np.any(std == 0):
        raise ValueError("zscore() is undefined for constant data (standard deviation 0)")
    return (x - x.mean(axis=axis, keepdims=True)) / std


def zscore_bug_1(x, ddof=0, axis=None):
    x = np.asarray(x, dtype=float)
    var = x.var(axis=axis, ddof=ddof, keepdims=True)
    if x.size == 0 or np.any(var == 0):
        raise ValueError("constant data")
    return (x - x.mean(axis=axis, keepdims=True)) / var


def zscore_bug_2(x, ddof=0, axis=None):
    x = np.asarray(x, dtype=float)
    std = x.std(axis=axis, ddof=ddof, keepdims=True)
    return (x - x.mean(axis=axis, keepdims=True)) / np.where(std == 0, 1.0, std)


def zscore_bug_3(x, ddof=0, axis=None):
    x = np.asarray(x, dtype=float)
    std = x.std(ddof=ddof)
    if x.size == 0 or std == 0:
        raise ValueError("constant data")
    return (x - x.mean()) / std

In [ ]:
def flag_outliers(x, threshold=3.0):
    """TODO: write the NumPy-style docstring here."""
    raise NotImplementedError("flag_outliers() is not written yet")


# Write your test functions here; they call zscore. For example:
# def test_something():
#     assert ...

my_zscore_tests = ...   # the list of your test functions

In [ ]:
with wb.attempt("2.31"):
    flagged_31 = flag_outliers([1, 1, 1, 1, 1, 1, 1, 1, 1, 10], threshold=2.5)
    verdict("2.31", np.ndim(flagged_31) == 1 and np.asarray(flagged_31).tolist() == [9],
            "flag_outliers trouve la valeur isolée (indice 9).",
            f"flag_outliers([1, 1, 1, 1, 1, 1, 1, 1, 1, 10], threshold=2.5) doit renvoyer l'array [9] "
            f"(un tableau d'indices, même pour un seul), pas {flagged_31!r}.")
    verdict("2.31", error_name(flag_outliers, [4, 4, 4]) == "ValueError",
            "des données constantes lèvent une ValueError.",
            "des données constantes doivent lever une ValueError (celle de zscore).")
    doc_31 = inspect.getdoc(flag_outliers) or ""
    missing_31 = [s for s in ("Parameters", "Returns", "Raises", "Examples")
                  if not re.search(rf"^{s}\n-{{3,}}$", doc_31, flags=re.MULTILINE)]
    verdict("2.31", not missing_31, "les quatre sections de la docstring sont là.", f"sections manquantes : {missing_31}")
    runner_31 = doctest.DocTestRunner(verbose=False)
    for test in doctest.DocTestFinder().find(flag_outliers, "flag_outliers",
                                             globs={"np": np, "flag_outliers": flag_outliers, "mylearn": mylearn}):
        runner_31.run(test)
    failed_31, attempted_31 = runner_31.summarize(verbose=False)
    verdict("2.31", attempted_31 >= 2 and failed_31 == 0, f"doctest : {attempted_31} exemples, tous justes.",
            f"doctest : {failed_31} exemple(s) faux sur {attempted_31} (détail au-dessus), ou moins de 2 exemples.")
if my_zscore_tests is ...:
    print("⏳ Ex 2.31 : tests pas encore écrits.")
else:
    good_31 = wb.run_pytest(my_zscore_tests, subject=zscore_ok, name="zscore")
    if good_31.ok and good_31.passed < len(my_zscore_tests):
        print("❌ Ex 2.31 : pytest n'a pas trouvé tous tes tests : chaque fonction de test doit s'appeler test_…")
    verdict("2.31", good_31.ok and good_31.passed >= 3, f"tes tests passent sur la version juste ({good_31.passed} cas).",
            "tes tests doivent tous passer sur la version juste, et être au moins 3 (voir le détail au-dessus).")
    if good_31.ok:
        for bug in [zscore_bug_1, zscore_bug_2, zscore_bug_3]:
            result = wb.run_pytest(my_zscore_tests, subject=bug, name="zscore", quiet=True)
            verdict("2.31", result.failed + result.errors > 0, f"{bug.__name__} est attrapé ({result.failed} test(s) en échec).",
                    f"tes tests passent sur {bug.__name__}, qui est faux : ajoute un test qui le fait échouer.")

### Ex 2.32 — Mêmes statistiques, autre dessin : fabrique ton quartet 🏆 ★★★ ⏱️ 60 min
**Objectif :** fabriquer un nuage de forme imposée qui a exactement les statistiques d'Anscombe.  
**Prérequis :** Ex 2.30, Ex 2.15, Ex 2.8 · fiche §2.9 (🕰️ Datasaurus) · **Fil rouge :** synthétique · **Parcours :** M, C

**Défi.** Ajoute un cinquième jeu au quartet, en forme de **cœur**, avec les statistiques d'Anscombe à 0,01 près :
- moyenne de $x$ : 9 ; moyenne de $y$ : 7,50 ;
- variance de $x$ : 11 et variance de $y$ : 4,125, **avec ddof = 1** (les chiffres des tableaux habituels, 2.12) ;
- corrélation : 0,816 (et donc la même droite, $y \approx 3 + 0{,}5\,x$).

`heart_points(40)` fournit 40 points $(h_x, h_y)$ le long d'un cœur. Écris `make_heart_dataset(n=40)`, qui renvoie `(x, y)`, deux arrays de $n$ nombres obtenus à partir de $h_x$ et $h_y$ par des **opérations affines** seulement (multiplier par un nombre, ajouter un nombre, additionner $h_x$ et $h_y$ multipliés par des nombres), pour que le cœur reste reconnaissable, étiré ou penché.

Une méthode, à suivre pas à pas (sans algèbre linéaire) :
1. standardise $h_x$ (ddof = 0) : $z_x$ a une moyenne 0 et un écart-type 1 ;
2. retire de $h_y$ sa partie « en ligne droite » avec $z_x$ : $e = h_y - \bar{h}_y - c\,z_x$, avec $c = \mathrm{Cov}(h_y, z_x)$ ; alors $\mathrm{Cov}(e, z_x) = 0$ (vérifie-le, à partir de la définition de la covariance ou numériquement), puis standardise $e$ en $z_e$. Ici, le cœur est symétrique : tu trouveras $c \approx 0$, mais l'étape compte pour une forme penchée ;
3. pour la corrélation voulue $r$, pose $z_y = r\,z_x + \sqrt{1 - r^2}\,z_e$ : alors $z_y$ a un écart-type 1 et $\mathrm{Corr}(z_x, z_y) = r$ (le démontrer est un bon exercice, en développant la définition de la covariance) ;
4. remets les centres et les échelles : $x = 9 + s_x\,z_x$ et $y = 7{,}5 + s_y\,z_y$, où $s_x$ et $s_y$ sont les écarts-types voulus… avec le bon ddof : $z_x$ a été standardisé avec ddof = 0, les variances visées le sont avec ddof = 1.

La vérification contrôle les cinq statistiques et que ton nuage est bien une image affine du cœur, puis trace les quatre jeux d'Anscombe et le tien, chacun avec sa droite : cinq dessins, une seule droite. Pour aller plus loin : une autre forme (un cercle, une étoile, tes initiales), ou 400 points au lieu de 40.

In [ ]:
x_123 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)
ANSCOMBE = {   # F. J. Anscombe, "Graphs in Statistical Analysis", 1973 (public domain)
    "I": (x_123, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    "II": (x_123, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    "III": (x_123, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    "IV": (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], dtype=float),
           np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}


def heart_points(n=40):
    """n points (hx, hy) along a heart-shaped curve (the classic parametric heart), in the order of the curve."""
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    hx = 16 * np.sin(t) ** 3
    hy = 13 * np.cos(t) - 5 * np.cos(2 * t) - 2 * np.cos(3 * t) - np.cos(4 * t)
    return hx, hy


def plot_quartet_and(x_new, y_new, name):
    """The four sets of ANSCOMBE and (x_new, y_new), each with its least-squares line."""
    fig, axes = plt.subplots(1, 5, figsize=(17, 3.6), sharex=True, sharey=True)
    for ax, (label, (x, y)) in zip(axes, [*ANSCOMBE.items(), (name, (x_new, y_new))]):
        slope, intercept = np.polyfit(x, y, 1)
        ax.scatter(x, y, s=14)
        ax.plot([0, 22], [intercept, intercept + 22 * slope], color="0.3")
        ax.set_title(f"{label}: y = {intercept:.2f} + {slope:.2f} x")
    plt.show()

In [ ]:
def make_heart_dataset(n=40):
    """(x, y): the heart of heart_points(n), transformed (affine operations only) to have Anscombe's statistics."""
    raise NotImplementedError("make_heart_dataset() is not written yet")

In [ ]:
with wb.attempt("2.32"):
    x_32, y_32 = (np.asarray(v, dtype=float) for v in make_heart_dataset(40))
    targets_32 = {"moyenne de x": (np.mean(x_32), 9.0), "moyenne de y": (np.mean(y_32), 7.5),
                  "variance de x (ddof = 1)": (np.var(x_32, ddof=1), 11.0),
                  "variance de y (ddof = 1)": (np.var(y_32, ddof=1), 4.125),
                  "corrélation": (np.corrcoef(x_32, y_32)[0, 1], 0.816)}
    for name, (value, target) in targets_32.items():
        verdict("2.32", abs(value - target) <= 0.01, f"{name} = {fr(value, 4)} (cible : {fr(target, 3)})",
                f"{name} = {fr(value, 4)}, loin de la cible {fr(target, 3)} (à 0,01 près)")
    hx_32, hy_32 = heart_points(40)
    design_32 = np.column_stack([np.ones(40), hx_32, hy_32])
    fits_32 = [np.linalg.lstsq(design_32, v, rcond=None)[0] for v in (x_32, y_32)]
    affine_32 = all(np.allclose(design_32 @ coef, v, atol=1e-6) for coef, v in zip(fits_32, (x_32, y_32)))
    determinant_32 = fits_32[0][1] * fits_32[1][2] - fits_32[0][2] * fits_32[1][1]
    verdict("2.32", len(x_32) == 40 and affine_32 and abs(determinant_32) > 1e-6,
            "ton nuage est une image affine du cœur : il reste un cœur, étiré ou penché.",
            "ton nuage n'est pas une image affine (non aplatie) du cœur, ou n'a pas 40 points : "
            "n'utilise que des multiplications, des additions et des combinaisons de h_x et h_y.")
    plot_quartet_and(x_32, y_32, "heart")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu choisir entre moyenne et médiane, et calculer variance et écart-type avec le bon diviseur ?
2. Sais-tu utiliser la règle 68-95-99,7 et dire quand elle ne s'applique pas ?
3. Sais-tu calculer une covariance et une corrélation, et dire ce qu'une corrélation ne prouve pas ?

**Pour aller plus loin** : *Seeing Theory* (Brown University) et le cours *Statistiques et probabilités* de Khan Academy, cités dans la fiche. La suite : le ch. 3 (probabilités conditionnelles, mesures de la qualité d'un classifieur), puis le bootstrap revient au ch. 14 (forêts aléatoires) et la matrice de covariance au ch. 12 (analyse en composantes principales).